In [1]:
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.stem.porter import PorterStemmer
from nltk.stem import WordNetLemmatizer

In [2]:
data = pd.read_csv(r"..\Data\Dataset.csv")
data.dropna(inplace=True)
data.head()

,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,9/25/2021,2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s3,TV Show,Ganglands,Julien Leclercq,France,9/24/2021,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,9/24/2021,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries",The arrival of a charismatic young priest brin...
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,9/22/2021,2021,TV-PG,91 min,"Children & Family Movies, Comedies",When the clever but socially-awkward Tetê join...
4,s8,Movie,Sankofa,Haile Gerima,United States,9/24/2021,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies","On a photo shoot in Ghana, an American model s..."


## ****Data Cleaning****
1. type & director: change case of all words to lower case and remove spaces.
2. listed_in: change case, remove repeating words like 'tv', 'shows', 'movies' also remove special characters
3. description: change case and remove special characters
4. will create a new column named tags and will concate all these cleaned column in it.

In [3]:
def lemma(text):
    return " ".join([WordNetLemmatizer().lemmatize(w) for w in text])

In [4]:
def clean_df(df_b: pd.DataFrame) -> pd.DataFrame:
    """Cleans i.e removing leading or trailing spaces, converting to lower case,
      creates tags by concating type, rating, description, listed_in columns"""

    req_columns = ["show_id", "type", "title", "director", "rating", "listed_in"]
    if "description" in df_b.columns:
        req_columns.append("description")
    
    df = df_b[req_columns].dropna().copy()
        
    try:
        for i in ["type", "rating"]:
            df[i] = df[i].str.lower().str.strip().str.replace(r"[^\w]", "", regex=True).str.split()

        df["director"] = df["director"].str.lower().str.strip().str.replace(" ", "").str.replace(",", " ").str.replace(r"[^\w\s]", "", regex=True).str.split()
        df["listed_in"] = df["listed_in"].str.lower().str.replace(r"\s+\b(tv|shows|movies)\b|[^\w\s]", "", regex=True).str.strip().str.split()

        tag_cols = ["listed_in", "director", "type", "rating"]

        if "description" in df.columns:
            df["description"] = df["description"].str.lower().str.replace("-", " ").str.replace(r"[^\w\s]", "", regex=True).str.split()
            tag_cols.insert(0, "description")

        df["tags"] = df[tag_cols].sum(axis=1).apply(lemma)
        df = df[["show_id", "title", "tags"]]
        return df
    
    except Exception as e:
        print(f"Failed to clean dataframe: {e}")
        raise

new_df = clean_df(data)

In [5]:
new_df["tags"]

0       a her father nears the end of his life filmmak...
1       to protect his family from a powerful drug lor...
2       the arrival of a charismatic young priest brin...
3       when the clever but socially awkward tet join ...
4       on a photo shoot in ghana an american model sl...
                              ...                        
8785    during the mongol invasion yunus emre leaf his...
8786    teen surfer zak storm is mysteriously transpor...
8787    strong willed middle class kashaf and carefree...
8788    friend mai oto and viks game at the park becom...
8789    with the mind of a human being and the body of...
Name: tags, Length: 8783, dtype: str

In [6]:
ps = PorterStemmer()
def stemmer(text):
    res = [ps.stem(i) for i in text.split()]
    return " ".join(res)

In [7]:
new_df["tags"][14]

'the star of bling empire discus the show success and play bling themed game then comic joel kim booster make his case for joining the cast movie krysiaplonka kristianmercado movie tvma'

In [8]:
tfidf = TfidfVectorizer(stop_words="english", max_features=5000)

vectors = tfidf.fit_transform(new_df["tags"]).toarray()

In [9]:
vectors[0]

array([0., 0., 0., ..., 0., 0., 0.], shape=(5000,))

In [10]:
similarity = cosine_similarity(vectors)

In [11]:
similarity

array([[1.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 1.        , 0.01327546, ..., 0.01504189, 0.02753795,
        0.01075438],
       [0.        , 0.01327546, 1.        , ..., 0.01518206, 0.00866441,
        0.03428018],
       ...,
       [0.        , 0.01504189, 0.01518206, ..., 1.        , 0.02227974,
        0.02495641],
       [0.        , 0.02753795, 0.00866441, ..., 0.02227974, 1.        ,
        0.0642041 ],
       [0.        , 0.01075438, 0.03428018, ..., 0.02495641, 0.0642041 ,
        1.        ]], shape=(8783, 8783))

In [12]:
def recommend(show):
    show_index = new_df[new_df["title"] == show].index[0]
    dist = similarity[show_index]
    shows_list = sorted(list(enumerate(dist)), reverse=True, key=lambda x: x[1])[1:6]

    rec_shows = []
    
    for i in shows_list:
        title = new_df["title"].iloc[i[0]]

        rec_shows.append(title)

    return rec_shows

In [29]:
l1 = sorted(list(enumerate(similarity[0])), reverse=True, key=lambda x: x[1])[1:6]

for i in l1:
    show_idx = i[0]
    print(new_df["title"].iloc[show_idx])

A Gray State
Black Snake Moan
Go Dog Go
Thor: Ragnarok
The Soul


In [30]:
import random

sh = random.choice(data["title"])

print("Chosen Show: ", sh)

recommend(sh)

Chosen Show:  Ingobernable


['Transformers Prime',
 "Edgar Rice Burroughs' Tarzan and Jane",
 'Heroes of Goo Jit Zu',
 'Power Rangers Samurai',
 'Miniforce: Super Dino Power']